<a href="https://colab.research.google.com/github/doanhailong8197-ai/spatial-gmm-vae/blob/main/Methods/run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = "https://github.com/doanhailong8197-ai/spatial-gmm-vae.git"
REPO_NAME = "spatial-gmm-vae"

if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone {REPO_URL}

%cd /content/{REPO_NAME}/Methods

!pip install -q scanpy anndata scikit-learn

import math
import random
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors

from preprocess import preprocess_pipeline
from model import (Encoder, Decoder, GMM_VAE,
                   pretrain_vae, initialize_gmm_parameters, train_joint_gmm_vae,
                   extract_mu, build_spatial_neighbors, compute_prior,
                   dual_rejection, predict_spatial)

warnings.filterwarnings("ignore")
sns.set_theme(style="white", context="notebook", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 100,
                     "axes.spines.top": False, "axes.spines.right": False})

CFG = dict(
    rna_path="/content/drive/MyDrive/data/seqFISH_40/adata_rna.h5ad",
    spa_path="/content/drive/MyDrive/data/seqFISH_40/adata_seqfish_40.h5ad",
    label_col="cell_type",
    n_top_genes=2000,
    batch_size=256,
    latent_dim=32,
    max_cells_plot=8000,
    seed=42,

    # --- Pretrain ---
    lr_pretrain=1e-3,
    epochs_pretrain=50,

    # --- Joint (Giai đoạn 2) ---
    lr_joint=5e-4,
    epochs_joint=60,
    lambda_1=0.2,          # trọng số GMM loss
    lambda_2=1.0,          # trọng số alignment loss
    align_temp=0.1,
    radial_weight=0.5,     # trọng số Deep CORAL
    pull_weight=1.0,
    ce_weight=0.5,
    pull_quantile=0.9,
    gate_quantile=0.95,    # cổng OOD hiệu chỉnh trên scRNA thật
    gate_softness=0.25,
    var_weight=0.5,
    var_ema=0.9,
    lambda_cls=1.0,

    # --- Suy luận / phát hiện OOD ---
    spatial_k=15,
    prior_alpha=0.5,
    chi2_conf=0.95,
    tau_H=0.5,
)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/spatial-gmm-vae/Methods
Device: cpu


# Read Data

In [ ]:
adata_rna = sc.read_h5ad(CFG["rna_path"])
adata_spa = sc.read_h5ad(CFG["spa_path"])

dl_rna, dl_spa, label_mapping = preprocess_pipeline(
    adata_rna, adata_spa,
    label_col=CFG["label_col"],
    n_top_genes=CFG["n_top_genes"],
    batch_size=CFG["batch_size"],
)

ds_rna, ds_spa = dl_rna.dataset, dl_spa.dataset
idx_to_label = {v: k for k, v in label_mapping.items()}
class_names = [idx_to_label[i] for i in range(len(idx_to_label))]

input_dim = ds_rna.data.shape[1]
n_clusters = len(label_mapping)
print(f"scRNA-seq: {ds_rna.data.shape[0]} cells | ST: {ds_spa.data.shape[0]} cells | "
      f"genes: {input_dim} | cell types (source): {n_clusters}")

# Khởi tạo mô hình

In [ ]:
latent_dim = CFG["latent_dim"]
hidden_dims = (256, 128, 64)

encoder = Encoder(input_dim, hidden_dims, latent_dim)
decoder_src = Decoder.from_encoder_dims(hidden_dims, latent_dim, input_dim)
decoder_tgt = Decoder.from_encoder_dims(hidden_dims, latent_dim, input_dim)

model = GMM_VAE(encoder, decoder_src, decoder_tgt,
                n_clusters=n_clusters, latent_dim=latent_dim,
                input_dim=input_dim).to(device)

optimizer_pre = torch.optim.Adam(model.parameters(), lr=CFG["lr_pretrain"])

print(f"hidden_dims: {hidden_dims}")
print(f"Tổng tham số: {sum(p.numel() for p in model.parameters()):,}")

# Train model

In [ ]:
hist_pre = pretrain_vae(model, dl_rna, optimizer_pre, CFG["epochs_pretrain"],
                        device, dataloader_spa=dl_spa)

plt.figure(figsize=(5, 3.2))
plt.plot(range(1, len(hist_pre) + 1), hist_pre, color="#3b6fb6", lw=2)
plt.xlabel("Epoch"); plt.ylabel("Reconstruction loss"); plt.title("Pre-training")
plt.tight_layout(); plt.show()

## K-Means trên mu của scRNA-seq để khởi tạo pi, mu_c, var_c

In [ ]:
info = initialize_gmm_parameters(model, dl_rna, n_clusters, device, seed=CFG["seed"])
label_to_cluster = info["label_to_cluster"]

assert (label_to_cluster >= 0).all(), "Có nhãn chưa được ghép cụm: kiểm tra n_clusters"
print("Ánh xạ nhãn -> cụm GMM:")
for lbl, clu in enumerate(label_to_cluster.tolist()):
    print(f"  {class_names[lbl]:<25} -> cụm {clu}")
print("pi:", np.round(info["pi"].cpu().numpy(), 3))

## huấn luyện chung. Tạo optimizer mới

In [ ]:
optimizer_joint = torch.optim.Adam(model.parameters(), lr=CFG["lr_joint"])

# Gọi MỘT lần cho toàn bộ epochs để src_var (EMA) và warm-up không bị reset
hist_joint = train_joint_gmm_vae(
    model, dl_rna, dl_spa, optimizer_joint,
    epochs=CFG["epochs_joint"],
    lambda_1=CFG["lambda_1"], lambda_2=CFG["lambda_2"],
    device=device,
    label_to_cluster=label_to_cluster,
    align_temp=CFG["align_temp"],
    radial_weight=CFG["radial_weight"],
    pull_weight=CFG["pull_weight"],
    ce_weight=CFG["ce_weight"],
    pull_quantile=CFG["pull_quantile"],
    gate_quantile=CFG["gate_quantile"],
    gate_softness=CFG["gate_softness"],
    var_weight=CFG["var_weight"],
    var_ema=CFG["var_ema"],
    lambda_cls=CFG["lambda_cls"],
)

hist_df = pd.DataFrame(hist_joint)
hist_df.index += 1
hist_df.index.name = "epoch"

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, cols in zip(axes, [["total"], ["recon", "gmm"], ["ot", "var", "pull"]]):
    for c in cols:
        ax.plot(hist_df.index, hist_df[c], lw=2, label=c)
    ax.set_xlabel("Epoch"); ax.legend()
axes[0].set_title("Total"); axes[1].set_title("Recon / GMM"); axes[2].set_title("Align (ot) / var / pull")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.2))
for ax, (col, color) in zip(axes, [("recon", "#3b6fb6"), ("gmm", "#d9822b"),
                                    ("domain", "#c0392b"), ("lambda_grl", "#2e8b57")]):
    ax.plot(hist_df.index + 1, hist_df[col], color=color, lw=2)
    ax.set_title(col); ax.set_xlabel("Epoch")
axes[2].axhline(math.log(2), ls="--", c="gray", lw=1)
axes[2].text(0.98, 0.9, "ln 2 (đoán ngẫu nhiên)", transform=axes[2].transAxes,
             ha="right", fontsize=8, color="gray")
plt.tight_layout(); plt.show()

In [ ]:
nbr = build_spatial_neighbors(adata_spa.obsm["spatial"], k=CFG["spatial_k"])

res = predict_spatial(model, ds_spa, device, nbr=nbr, prior_mode="spatial",
                      alpha=CFG["prior_alpha"], tau_H=CFG["tau_H"],
                      chi2_conf=CFG["chi2_conf"])

pred = res["pseudo_labels"].cpu().numpy()   # -1 = Unknown (OOD)

## Trích xuất không gian ẩn

In [ ]:
@torch.no_grad()
def extract_latent(model, dataset, batch_size, device):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)
    zs, ys, gs = [], [], []
    for x, y, _ in loader:
        mu, _ = model.encoder(x.to(device))
        zs.append(mu.cpu())
        gs.append(model.get_gamma(mu).cpu())
        ys.append(y)
    return torch.cat(zs).numpy(), torch.cat(ys).numpy(), torch.cat(gs).numpy()

model.eval()
Z_rna, y_rna, gamma_rna = extract_latent(model, ds_rna, CFG["batch_size"], device)
Z_spa, y_spa, gamma_spa = extract_latent(model, ds_spa, CFG["batch_size"], device)
print("Z_rna:", Z_rna.shape, "| Z_spa:", Z_spa.shape)

# Evaluate

In [ ]:
rng = np.random.default_rng(CFG["seed"])
n_plot = min(CFG["max_cells_plot"], len(Z_rna), len(Z_spa))
idx_r = np.sort(rng.choice(len(Z_rna), n_plot, replace=False))
idx_s = np.sort(rng.choice(len(Z_spa), n_plot, replace=False))

Z_plot = np.vstack([Z_rna[idx_r], Z_spa[idx_s]])
domain = np.array(["scRNA-seq"] * n_plot + ["Spatial (ST)"] * n_plot)
is_rna = domain == "scRNA-seq"

ad = sc.AnnData(Z_plot.astype(np.float32))
sc.pp.neighbors(ad, n_neighbors=15, use_rep="X")
sc.tl.umap(ad, random_state=CFG["seed"])
emb_umap = ad.obsm["X_umap"]

emb_tsne = TSNE(n_components=2, perplexity=30, init="pca",
                random_state=CFG["seed"], n_jobs=-1).fit_transform(Z_plot)

embeddings = {"UMAP": emb_umap, "t-SNE": emb_tsne}

In [ ]:
def style_axis(ax, title, name):
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlabel(f"{name} 1"); ax.set_ylabel(f"{name} 2")
    ax.set_xticks([]); ax.set_yticks([])

def scatter_cat(ax, emb, labels, palette, s=5, alpha=0.8):
    for lab, color in palette.items():
        m = labels == lab
        ax.scatter(emb[m, 0], emb[m, 1], s=s, c=[color], alpha=alpha,
                   linewidths=0, rasterized=True, label=lab)

domain_palette = {"scRNA-seq": "#3b6fb6", "Spatial (ST)": "#e07b39"}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
order = rng.permutation(len(domain))
for ax, (name, emb) in zip(axes, embeddings.items()):
    for dom, color in domain_palette.items():
        pass
    ax.scatter(emb[order, 0], emb[order, 1], s=5, linewidths=0, rasterized=True, alpha=0.7,
               c=[domain_palette[d] for d in domain[order]])
    style_axis(ax, f"Domain alignment ({name})", name)

handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=k, markersize=7)
           for k, c in domain_palette.items()]
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(); plt.savefig("fig1_domain_alignment.png", bbox_inches="tight"); plt.show()

In [ ]:
labels_rna_plot = np.array([class_names[i] for i in y_rna[idx_r]])

type_palette = dict(zip(class_names, sns.color_palette("husl", len(class_names))
                        if len(class_names) > 20 else sns.color_palette("tab20", len(class_names))))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for ax, (name, emb) in zip(axes, embeddings.items()):
    scatter_cat(ax, emb[is_rna], labels_rna_plot, type_palette)
    style_axis(ax, f"scRNA-seq theo Ground Truth ({name})", name)
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
               markerscale=3, fontsize=9, title="Cell type")
plt.tight_layout(); plt.savefig("fig2_source_celltype.png", bbox_inches="tight"); plt.show()

In [ ]:
g = gamma_spa[idx_s]
comp = g.argmax(1)
entropy = -(g * np.log(g + 1e-10)).sum(1)
cluster_to_label = {int(c): class_names[l] for l, c in enumerate(label_to_cluster.tolist())}

# Tạo lưới 2x2: Hàng 1 (GMM), Hàng 2 (Entropy) | Cột 1 (UMAP), Cột 2 (t-SNE)
fig, axes = plt.subplots(2, 2, figsize=(15, 10.5))

# --- HÀNG 1: ST theo thành phần GMM ---
for idx, (name, emb_full) in enumerate(embeddings.items()):
    ax = axes[0, idx]
    emb = emb_full[~is_rna]

    for c in np.unique(comp):
        m = comp == c
        cell_type = cluster_to_label.get(c, '?')
        # LẤY MÀU TỪ TYPE_PALETTE CỦA Cell 1 ĐỂ ĐỒNG BỘ
        color = type_palette.get(cell_type, '#333333')

        ax.scatter(emb[m, 0], emb[m, 1], s=5, c=[color], linewidths=0, alpha=0.8,
                   rasterized=True, label=f"{c}: {cell_type}" if idx == 1 else "")

    style_axis(ax, f"ST theo thành phần GMM ({name})", name)

# Đặt legend cho Hàng 1 (nằm ở rìa ngoài cùng bên phải của đồ thị t-SNE)
axes[0, 1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
                  markerscale=3, fontsize=9, title="Cụm: loại nguồn tương ứng")

# --- HÀNG 2: ST theo Entropy ---
for idx, (name, emb_full) in enumerate(embeddings.items()):
    ax = axes[1, idx]
    emb = emb_full[~is_rna]

    sc_ = ax.scatter(emb[:, 0], emb[:, 1], s=5, c=entropy, cmap="magma",
                     linewidths=0, alpha=0.85, rasterized=True)
    style_axis(ax, f"ST theo Entropy của γ ({name})", name)
    cb = plt.colorbar(sc_, ax=ax, fraction=0.046, pad=0.02)
    cb.set_label("Entropy")

plt.tight_layout(); plt.savefig("fig3_target_structure.png", bbox_inches="tight"); plt.show()

In [ ]:
labels_spa_plot = adata_spa.obs[CFG["label_col"]].values[idx_s]
new_labels = set(labels_spa_plot) - set(type_palette.keys())
if new_labels:
    extra_colors = sns.color_palette("dark", len(new_labels))
    for label, color in zip(new_labels, extra_colors):
        type_palette[label] = color

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

for ax, (name, emb) in zip(axes, embeddings.items()):
    scatter_cat(ax, emb[~is_rna], labels_spa_plot, type_palette)
    style_axis(ax, f"ST theo Ground Truth ({name})", name)

axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
               markerscale=3, fontsize=9, title="ST Ground Truth")

plt.tight_layout()
plt.savefig("fig4_target_groundtruth.png", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np

labels_spa_original = adata_spa.obs[CFG["label_col"]].values[idx_s]

known_labels = set(class_names)

labels_spa_grouped = np.array([
    label if label in known_labels else "Unknown"
    for label in labels_spa_original
])

palette_unknown = {label: type_palette[label] for label in known_labels}
palette_unknown["Unknown"] = "#555555" # Màu xám đậm (Dark Gray) để dễ nhận diện

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

for ax, (name, emb) in zip(axes, embeddings.items()):
    emb_spa = emb[~is_rna]

    scatter_cat(ax, emb_spa, labels_spa_grouped, palette_unknown)
    style_axis(ax, f"ST Ground Truth (OOD masked) - {name}", name)

axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
               markerscale=3, fontsize=9, title="ST Ground Truth\n(Known vs Unknown)")

plt.tight_layout()
plt.savefig("fig5_target_groundtruth_unknown.png", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

K = gamma_rna.shape[1]
cluster_names = np.empty(K, dtype=object)
cluster_to_lbl = np.empty(K, dtype=int)
for lbl, c in enumerate(label_to_cluster.tolist()):
    cluster_names[c] = class_names[lbl]
    cluster_to_lbl[c] = lbl

def to_names(cluster, accept):
    return np.where(accept, cluster_names[cluster], "Unknown")

def to_idx(cluster, accept):
    return np.where(accept, cluster_to_lbl[cluster], -1)

def open_set_metrics(pred, y_true):
    known, unk = y_true >= 0, y_true < 0
    kacc = (pred[known] == y_true[known]).mean() if known.any() else np.nan
    uacc = (pred[unk] == -1).mean() if unk.any() else np.nan
    tacc = (pred == y_true).mean()
    h_score = (2.0 * kacc * uacc) / (kacc + uacc + 1e-12)
    return {
        "accepted": (pred >= 0).mean(),
        "KACC": kacc,
        "UACC": uacc,
        "TACC": tacc,
        "H-score": h_score
    }

def spherical_sinkhorn_universal_cascade(
    gamma_target, Z_target, Z_source, gamma_source, model,
    source_geom_pct=95.0,
    source_ent_pct=88.0,         # Khớp với Hình 8: Lọc bỏ dải điểm Đỏ & Xám nằm ở vùng Entropy > 0.30
    cosine_blend=0.50,           # Kết hợp 50% khoảng cách Cầu (Cosine) + 50% Mahalanobis để khắc phục nhiễu Hình 7
    debias_hubness=True,         # Khử hiệu ứng Hubness (ngăn Neural crest hút OOD & cứu Endothelium/Spinal cord)
    hub_damping=0.45,            # Mức độ kìm hãm các cụm bị phình bất thường trên miền đích
    coupling_strength=0.40       # Siết chặt Entropy cho các điểm nằm xa tâm cụm
):
    """
    Spherical-Sinkhorn Universal Cascade (Tổng quát cho mọi bộ dữ liệu scRNA-seq -> ST):
    1. Kết hợp khoảng cách Góc trên mặt cầu (Spherical Cosine) và Mahalanobis để chống suy biến phương sai 32 chiều.
    2. Khử hiện tượng Hubness / Mode Collapse bằng cân bằng tỷ trọng biên (Marginal Hubness Damping):
       - Ngăn cụm nhỏ (Neural crest) phình to nuốt đám mây Unknown ở trung tâm.
       - Khôi phục biên quyết định cho các cụm bị chèn ép (Endothelium, Spinal cord).
    3. Lọc Cascade khớp nối (Coupled Hybrid Geometry + Tight Entropy) dựa trên cấu trúc Không gian Quyết định.
    """
    cluster_pred_source = gamma_source.argmax(axis=1)
    raw_pred_target = gamma_target.argmax(axis=1)

    mu_c = model.mu_c.detach().cpu().numpy().astype(np.float64)
    var_c = model.var_c.detach().cpu().numpy().astype(np.float64)
    inv_var = 1.0 / var_c

    Z_t = Z_target.astype(np.float64)
    Z_s = Z_source.astype(np.float64)

    # 1. Tính khoảng cách Mahalanobis chuẩn hóa theo phân vị 95% của từng cụm nguồn
    d2_s_maha = (
        (Z_s ** 2) @ inv_var.T
        - (2.0 * Z_s) @ (mu_c * inv_var).T
        + ((mu_c ** 2) * inv_var).sum(axis=1)
    )
    d2_s_maha = np.clip(d2_s_maha, 0.0, None)

    d2_t_maha = (
        (Z_t ** 2) @ inv_var.T
        - (2.0 * Z_t) @ (mu_c * inv_var).T
        + ((mu_c ** 2) * inv_var).sum(axis=1)
    )
    d2_t_maha = np.clip(d2_t_maha, 0.0, None)

    # 2. Tính khoảng cách Góc (Spherical Cosine Distance: 1 - cos(z, mu_c)) không phụ thuộc phương sai
    # Chuẩn hóa L2 sau khi trừ tâm toàn cục của từng miền (Domain Centering để khử Translation Shift)
    Z_s_centered = Z_s - np.median(Z_s, axis=0, keepdims=True)
    Z_t_centered = Z_t - np.median(Z_t, axis=0, keepdims=True)
    mu_c_centered = mu_c - np.median(Z_s, axis=0, keepdims=True)

    Z_s_norm = Z_s_centered / (np.linalg.norm(Z_s_centered, axis=1, keepdims=True) + 1e-12)
    Z_t_norm = Z_t_centered / (np.linalg.norm(Z_t_centered, axis=1, keepdims=True) + 1e-12)
    mu_c_norm = mu_c_centered / (np.linalg.norm(mu_c_centered, axis=1, keepdims=True) + 1e-12)

    d_s_cos = np.clip(1.0 - (Z_s_norm @ mu_c_norm.T), 0.0, 2.0)
    d_t_cos = np.clip(1.0 - (Z_t_norm @ mu_c_norm.T), 0.0, 2.0)

    # 3. Chuẩn hóa cả Mahalanobis và Cosine về thang phân vị 95% của từng cụm trên tập Nguồn
    maha_thr_cls = np.zeros(K)
    cos_thr_cls = np.zeros(K)
    fb_maha = np.percentile(d2_s_maha.min(axis=1), source_geom_pct)
    fb_cos = np.percentile(d_s_cos.min(axis=1), source_geom_pct)

    for k in range(K):
        mask_k = (cluster_pred_source == k)
        if mask_k.sum() >= 5:
            maha_thr_cls[k] = np.percentile(d2_s_maha[mask_k, k], source_geom_pct)
            cos_thr_cls[k] = np.percentile(d_s_cos[mask_k, k], source_geom_pct)
        else:
            maha_thr_cls[k] = fb_maha
            cos_thr_cls[k] = fb_cos

    r_s_maha = d2_s_maha / (maha_thr_cls[None, :] + 1e-12)
    r_t_maha = d2_t_maha / (maha_thr_cls[None, :] + 1e-12)

    r_s_cos = d_s_cos / (cos_thr_cls[None, :] + 1e-12)
    r_t_cos = d_t_cos / (cos_thr_cls[None, :] + 1e-12)

    # Khoảng cách hình học lai (Hybrid Spherical-Mahalanobis Ratio)
    r_s_hybrid = (1.0 - cosine_blend) * r_s_maha + cosine_blend * r_s_cos
    r_t_hybrid = (1.0 - cosine_blend) * r_t_maha + cosine_blend * r_t_cos

    # 4. KHỬ HIỆU ỨNG ĐIỂM HÚT (Hubness Damping) & KHÔI PHỤC CỤM BỊ CHÈN ÉP (Endothelium, Spinal cord)
    # Đo tần suất kỳ vọng trên tập nguồn vs tần suất thực tế trên tập đích (chỉ tính trên các điểm có Entropy thấp)
    entropy_source = -np.sum(gamma_source * np.log(gamma_source + 1e-12), axis=1)
    entropy_target = -np.sum(gamma_target * np.log(gamma_target + 1e-12), axis=1)

    if debias_hubness:
        src_freq = gamma_source.mean(axis=0) + 1e-4
        # Chỉ tính tần suất đích trên các điểm có độ tự tin tương đối để ước lượng mức độ phình cụm (Hubness)
        confident_t = entropy_target <= np.median(entropy_target)
        tgt_freq = gamma_target[confident_t].mean(axis=0) + 1e-4 if confident_t.sum() > 50 else gamma_target.mean(axis=0) + 1e-4

        # Hệ số phạt những cụm phình to bất thường (như Neural crest) và nâng đỡ cụm bị chèn ép (như Endothelium)
        hub_ratio = tgt_freq / src_freq
        col_scale = (1.0 / (hub_ratio ** hub_damping))
        col_scale = np.clip(col_scale, 0.35, 2.8)

        # Kết hợp xác suất hậu nghiệm đã khử Hubness với nhân hình học mặt cầu exp(-0.5 * r_t_cos)
        calibrated_gamma = (gamma_target * col_scale[None, :]) * np.exp(-0.5 * r_t_cos)
        calibrated_gamma = calibrated_gamma / (calibrated_gamma.sum(axis=1, keepdims=True) + 1e-12)

        # CHỈ cho phép đổi nhãn sang cụm mới nếu cụm mới nằm trong Top-2 ứng viên của gamma gốc HOẶC có khoảng cách cầu cực gần
        top2_idx = np.argsort(gamma_target, axis=1)[:, -2:]
        new_pred = calibrated_gamma.argmax(axis=1)
        is_in_top2 = (new_pred == top2_idx[:, 0]) | (new_pred == top2_idx[:, 1]) | (r_t_cos[np.arange(len(Z_t)), new_pred] <= 0.85)
        cluster_pred_target = np.where(is_in_top2, new_pred, raw_pred_target)
    else:
        cluster_pred_target = raw_pred_target

    # 5. TẦNG 1: Lọc OOD bằng Khoảng cách Lai (Spherical-Mahalanobis) + Kiểm soát Phình cụm
    # Lưu ý: Khi kiểm tra điều kiện lọt lưới OOD, ta kiểm tra nghiêm ngặt trên cả nhãn gốc lẫn nhãn đã hiệu chỉnh
    r_s_assigned = r_s_hybrid[np.arange(len(Z_s)), cluster_pred_source]
    r_t_assigned = r_t_hybrid[np.arange(len(Z_t)), cluster_pred_target]
    r_t_raw_assigned = r_t_hybrid[np.arange(len(Z_t)), raw_pred_target]

    r_t_sorted = np.sort(r_t_hybrid, axis=1)
    r_t_min1 = r_t_sorted[:, 0]
    r_t_min2 = r_t_sorted[:, 1]

    # Điều kiện đồng thuận & tách biệt hình học trên không gian lai
    pass_geom_cons = (r_t_assigned <= (r_t_min1 * 1.15 + 0.05)) & (r_t_raw_assigned <= (r_t_min1 * 1.25 + 0.10))
    pass_separation = r_t_min1 <= (r_t_min2 * 0.92)

    # Tự động ước lượng Margin trên thang sqrt(r_hybrid)
    reliable_pool = pass_geom_cons & (entropy_target <= 0.35)
    if reliable_pool.sum() < 100:
        reliable_pool = pass_geom_cons

    sqrt_r_tgt = np.sqrt(r_t_assigned[reliable_pool])
    med_t = np.median(sqrt_r_tgt)
    iqr_t = np.percentile(sqrt_r_tgt, 75.0) - np.percentile(sqrt_r_tgt, 25.0)
    global_margin = float(np.clip((med_t + 1.15 * (iqr_t / 1.349)) ** 2, 1.8, 4.2))

    # Siết chặt Margin đối với những cụm bị phình to trên miền đích (chặn đám mây Tím ở trung tâm Hình 5)
    src_counts_ratio = np.bincount(cluster_pred_source, minlength=K) / float(len(Z_s))
    tgt_counts_ratio = np.bincount(raw_pred_target, minlength=K) / float(len(Z_t))

    cluster_margins = np.full(K, global_margin)
    for k in range(K):
        if tgt_counts_ratio[k] > 1.5 * src_counts_ratio[k]:
            # Cụm phình to do hút OOD (như Neural crest: 16% vs 2%) -> Phạt giảm bán kính của cụm đó!
            penalty = ((src_counts_ratio[k] + 0.01) / (tgt_counts_ratio[k] + 0.01)) ** 0.40
            cluster_margins[k] = np.clip(global_margin * penalty, 1.2, global_margin)

    st_margins = cluster_margins[cluster_pred_target]
    pass_dist = (r_t_assigned <= st_margins) & (r_t_raw_assigned <= cluster_margins[raw_pred_target] * 1.10)
    accept_stage1 = pass_dist & pass_geom_cons & pass_separation

    # 6. TẦNG 2: Lọc Entropy Chặt theo Không gian Quyết định (Hình 8)
    # Trên Hình 8, các điểm Known đúng nhãn có Entropy rất thấp (< 0.25), trong khi điểm Sai nhãn & OOD nằm ở > 0.30
    class_ent_thr = np.zeros(K)
    fb_ent = np.percentile(entropy_source, source_ent_pct)
    for k in range(K):
        mask_k = (cluster_pred_source == k) & (r_s_assigned <= 1.0)
        if mask_k.sum() >= 5:
            class_ent_thr[k] = np.percentile(entropy_source[mask_k], source_ent_pct)
        else:
            class_ent_thr[k] = fb_ent

    base_ent_thr = np.clip(class_ent_thr[cluster_pred_target] * 1.10, 0.22, 0.48)

    # Đường biên khớp nối: Càng xa tâm cụm (rel_dist -> 1), ngưỡng Entropy càng hạ thấp để cắt đám mây Đỏ & Xám ở Hình 8
    rel_dist = np.clip(r_t_assigned / (st_margins + 1e-12), 0.0, 1.0)
    dynamic_ent_thr = np.clip(
        base_ent_thr * (1.0 + coupling_strength * (1.0 - 2.0 * rel_dist)),
        0.15, 0.52
    )

    accept_stage2 = entropy_target <= dynamic_ent_thr
    accept_final = accept_stage1 & accept_stage2

    # Trả về khoảng cách tương đương trên thang đo cũ để vẽ Hình 7 & Hình 8
    d2_assigned_out = d2_t_maha[np.arange(len(Z_t)), cluster_pred_target]

    info = {
        "effective_margin": round(float(st_margins.mean()), 3),
        "margin_min_max": (round(float(cluster_margins.min()), 2), round(float(cluster_margins.max()), 2)),
        "mean_ent_thr": round(float(dynamic_ent_thr.mean()), 4),
        "reject_stage1_maha": round(float((~accept_stage1).mean()), 4),
        "reject_stage2_ent": round(float((accept_stage1 & ~accept_stage2).mean()), 4),
        "reassigned_pct": round(float((cluster_pred_target != raw_pred_target).mean()), 4)
    }

    return cluster_pred_target, np.sqrt(d2_assigned_out), entropy_target, accept_final, info

# --- 1. CHẠY PHƯƠNG PHÁP SPHERICAL-SINKHORN UNIVERSAL CASCADE ---
cl_spa, d_mah, ent_spa, acc_spa, info_main = spherical_sinkhorn_universal_cascade(
    gamma_spa, Z_spa, Z_rna, gamma_rna, model,
    source_geom_pct=95.0, source_ent_pct=88.0,
    cosine_blend=0.50, debias_hubness=True, hub_damping=0.45
)

pred_names = to_names(cl_spa, acc_spa)
res_refined = open_set_metrics(to_idx(cl_spa, acc_spa), y_spa)

print(f"---> SPHERICAL-SINKHORN UNIVERSAL CASCADE (Mean Margin = {info_main['effective_margin']} {info_main['margin_min_max']}, Mean_Ent_Thr = {info_main['mean_ent_thr']})\n")
print(f"ST: Chấp nhận chung cuộc {acc_spa.mean():.1%} | Tổng bị loại (Unknown) {(1.0 - acc_spa.mean()):.1%}")
print(f"  ├── Tỷ lệ tế bào được hiệu chỉnh Hubness (cứu Endothelium/Spinal cord): {info_main['reassigned_pct']:.1%}")
print(f"  ├── Bị loại ở Tầng 1 (Spherical-Maha Hybrid + Hub Penalty):            {info_main['reject_stage1_maha']:.1%}")
print(f"  └── Bị loại ở Tầng 2 (Tight Coupled Entropy):                          {info_main['reject_stage2_ent']:.1%}\n")

print("Số lượng tế bào phân bổ theo nhãn (ST):")
print(pd.Series(pred_names).value_counts().to_string())

print("\n--- Đánh giá kết quả (Spherical-Sinkhorn Universal Cascade) ---")
print({k: round(float(v), 4) for k, v in res_refined.items()})

# --- 2. BẢNG ĐỐI CHIẾU TÁC ĐỘNG CỦA HÌNH HỌC MẶT CẦU (COSINE BLEND) & KHỬ HUBNESS ---
bench = {}

# (a) Mốc trước đó (Chỉ dùng Mahalanobis, không dùng Cosine, không khử Hubness)
c, _, _, a, inf = spherical_sinkhorn_universal_cascade(
    gamma_spa, Z_spa, Z_rna, gamma_rna, model,
    cosine_blend=0.0, debias_hubness=False
)
m_prev = open_set_metrics(to_idx(c, a), y_spa)
m_prev["Margin"] = inf["effective_margin"]
m_prev["Reassigned%"] = inf["reassigned_pct"]
bench["1. Mahalanobis Only (No Cosine, No HubDebias)"] = m_prev

# (b) Thêm khoảng cách Mặt cầu + Penalty chống phình cụm (Giữ nguyên argmax)
for c_blend in (0.35, 0.50, 0.65):
    c, _, _, a, inf = spherical_sinkhorn_universal_cascade(
        gamma_spa, Z_spa, Z_rna, gamma_rna, model,
        cosine_blend=c_blend, debias_hubness=False
    )
    m_sph = open_set_metrics(to_idx(c, a), y_spa)
    m_sph["Margin"] = inf["effective_margin"]
    m_sph["Reassigned%"] = inf["reassigned_pct"]
    bench[f"2. Spherical-Maha + HubPenalty (cos={c_blend}, RawArgmax)"] = m_sph

# (c) Kết hợp đầy đủ: Spherical-Mahalanobis + Khử Hubness khôi phục Endothelium/Spinal cord
for c_blend in (0.35, 0.50, 0.65):
    for damp in (0.30, 0.45, 0.60):
        c, _, _, a, inf = spherical_sinkhorn_universal_cascade(
            gamma_spa, Z_spa, Z_rna, gamma_rna, model,
            cosine_blend=c_blend, debias_hubness=True, hub_damping=damp
        )
        m_full = open_set_metrics(to_idx(c, a), y_spa)
        m_full["Margin"] = inf["effective_margin"]
        m_full["Reassigned%"] = inf["reassigned_pct"]
        bench[f"3. Full Spherical-Sinkhorn (cos={c_blend}, damp={damp})"] = m_full

print("\n--- Bảng đối chiếu: Mahalanobis thuần túy vs. Spherical-Sinkhorn Universal Cascade ---")
display(pd.DataFrame(bench).T.round(4))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# 1. Ánh xạ chuẩn giữa chỉ số lớp (0..K-1) và tên lớp
K = len(class_names)
cluster_names = np.array(class_names, dtype=object)
cluster_to_lbl = np.arange(K, dtype=int)

def to_names(cluster, accept):
    return np.where(accept, cluster_names[cluster], "Unknown")

def to_idx(cluster, accept):
    return np.where(accept, cluster_to_lbl[cluster], -1)

def open_set_metrics(pred, y_true):
    known, unk = y_true >= 0, y_true < 0
    kacc = (pred[known] == y_true[known]).mean() if known.any() else np.nan
    uacc = (pred[unk] == -1).mean() if unk.any() else np.nan
    tacc = (pred == y_true).mean()
    h_score = (2.0 * kacc * uacc) / (kacc + uacc + 1e-12)
    return {
        "accepted": (pred >= 0).mean(),
        "KACC": kacc,
        "UACC": uacc,
        "TACC": tacc,
        "H-score": h_score
    }

def supervised_prototype_spherical_cascade(
    Z_target, Z_source, y_source,
    source_geom_pct=95.0,
    source_ent_pct=90.0,
    cosine_blend=0.65,           # Kết hợp 65% khoảng cách Mặt cầu (Cosine) + 35% Mahalanobis có giám sát
    var_shrinkage=0.20,          # Điều chuẩn phương sai Ledoit-Wolf giúp ổn định các cụm nhỏ (Endothelium, Neural crest)
    temperature=0.15,            # Nhiệt độ Softmax trên khoảng cách chuẩn hóa để tạo phân phối xác suất sắc nét
    sep_ratio_thr=0.90,          # Ngưỡng tách biệt giữa cụm gần nhất và cụm gần nhì (lọc đám mây OOD trung tâm)
    coupling_strength=0.35       # Khớp nối khoảng cách và Entropy
):
    """
    Supervised-Prototype Spherical-Mahalanobis Cascade (Tổng quát cho mọi bộ dữ liệu):
    - Khắc phục hoàn toàn lỗi KMeans làm mất tâm cụm Endothelium, Spinal cord, Intermediate mesoderm
      bằng cách ước lượng trực tiếp Prototype (mu_k, var_k) từ (Z_rna, y_rna).
    - Kết hợp khoảng cách Góc mặt cầu (Spherical Cosine) và Mahalanobis chuẩn hóa phân vị.
    - Không cần bất kỳ tham số căn chỉnh thủ công nào theo nhãn đích y_spa.
    """
    Z_t = Z_target.astype(np.float64)
    Z_s = Z_source.astype(np.float64)
    y_s = np.asarray(y_source, dtype=int)
    D = Z_s.shape[1]

    # 1. Tính Tâm cụm (mu_sup) và Phương sai (var_sup) CÓ GIÁM SÁT trực tiếp từ (Z_rna, y_rna)
    # Dùng Median (trung vị) và MAD để kháng nhiễu ngoại lai trong từng lớp nguồn
    mu_sup = np.zeros((K, D), dtype=np.float64)
    var_sup_raw = np.zeros((K, D), dtype=np.float64)

    global_var = np.var(Z_s, axis=0) + 1e-6
    for k in range(K):
        mask_k = (y_s == k)
        if mask_k.sum() >= 5:
            Z_s_k = Z_s[mask_k]
            mu_sup[k] = np.median(Z_s_k, axis=0)
            # Ước lượng phương sai bền vững kết hợp phương sai mẫu và MAD
            mad_k = np.median(np.abs(Z_s_k - mu_sup[k]), axis=0) * 1.4826
            emp_var_k = 0.5 * np.var(Z_s_k, axis=0) + 0.5 * (mad_k ** 2)
            var_sup_raw[k] = np.maximum(emp_var_k, 1e-4)
        else:
            mu_sup[k] = np.median(Z_s, axis=0)
            var_sup_raw[k] = global_var

    # Điều chuẩn phương sai (Shrinkage về trung vị các cụm) để tránh cụm nhỏ có phương sai quá hẹp hoặc quá dẹt
    med_var_across_k = np.median(var_sup_raw, axis=0, keepdims=True)
    var_sup = (1.0 - var_shrinkage) * var_sup_raw + var_shrinkage * med_var_across_k
    inv_var = 1.0 / (var_sup + 1e-8)

    # 2. Tính khoảng cách Mahalanobis tới các tâm cụm có giám sát
    d2_s_maha = (
        (Z_s ** 2) @ inv_var.T
        - (2.0 * Z_s) @ (mu_sup * inv_var).T
        + ((mu_sup ** 2) * inv_var).sum(axis=1)
    )
    d2_s_maha = np.clip(d2_s_maha, 0.0, None)

    d2_t_maha = (
        (Z_t ** 2) @ inv_var.T
        - (2.0 * Z_t) @ (mu_sup * inv_var).T
        + ((mu_sup ** 2) * inv_var).sum(axis=1)
    )
    d2_t_maha = np.clip(d2_t_maha, 0.0, None)

    # 3. Tính khoảng cách Góc mặt cầu (Spherical Cosine Distance) sau khi căn tâm miền
    center_s = np.median(Z_s, axis=0, keepdims=True)
    center_t = np.median(Z_t, axis=0, keepdims=True)

    Z_s_c = Z_s - center_s
    # Căn tâm nhẹ giữa 2 miền (50% tâm nguồn + 50% tâm đích) để tránh kéo đám mây OOD làm lệch tâm
    Z_t_c = Z_t - (0.5 * center_s + 0.5 * center_t)
    mu_sup_c = mu_sup - center_s

    Z_s_norm = Z_s_c / (np.linalg.norm(Z_s_c, axis=1, keepdims=True) + 1e-12)
    Z_t_norm = Z_t_c / (np.linalg.norm(Z_t_c, axis=1, keepdims=True) + 1e-12)
    mu_norm = mu_sup_c / (np.linalg.norm(mu_sup_c, axis=1, keepdims=True) + 1e-12)

    d_s_cos = np.clip(1.0 - (Z_s_norm @ mu_norm.T), 0.0, 2.0)
    d_t_cos = np.clip(1.0 - (Z_t_norm @ mu_norm.T), 0.0, 2.0)

    # 4. Chuẩn hóa khoảng cách theo phân vị 95% của từng lớp thật trên tập Nguồn (RNA)
    maha_thr_cls = np.zeros(K)
    cos_thr_cls = np.zeros(K)
    fb_maha = np.percentile(d2_s_maha.min(axis=1), source_geom_pct)
    fb_cos = np.percentile(d_s_cos.min(axis=1), source_geom_pct)

    for k in range(K):
        mask_k = (y_s == k)
        if mask_k.sum() >= 5:
            maha_thr_cls[k] = np.percentile(d2_s_maha[mask_k, k], source_geom_pct)
            cos_thr_cls[k] = np.percentile(d_s_cos[mask_k, k], source_geom_pct)
        else:
            maha_thr_cls[k] = fb_maha
            cos_thr_cls[k] = fb_cos

    r_s_maha = d2_s_maha / (maha_thr_cls[None, :] + 1e-12)
    r_t_maha = d2_t_maha / (maha_thr_cls[None, :] + 1e-12)

    r_s_cos = d_s_cos / (cos_thr_cls[None, :] + 1e-12)
    r_t_cos = d_t_cos / (cos_thr_cls[None, :] + 1e-12)

    # Khoảng cách lai không thứ nguyên r_hybrid (r = 1.0 là biên 95% của từng loại tế bào trên RNA)
    r_s_hybrid = (1.0 - cosine_blend) * r_s_maha + cosine_blend * r_s_cos
    r_t_hybrid = (1.0 - cosine_blend) * r_t_maha + cosine_blend * r_t_cos

    # 5. Tái tạo Phân phối Xác suất Hậu nghiệm Cân bằng Lớp (Class-Balanced Posterior gamma_sup)
    # Dùng prior đồng đều (Uniform Prior) để các cụm nhỏ như Endothelium, Spinal cord không bị cụm lớn nuốt
    logits_s = -r_s_hybrid / temperature
    logits_s = logits_s - logits_s.max(axis=1, keepdims=True)
    gamma_s_sup = np.exp(logits_s) / (np.exp(logits_s).sum(axis=1, keepdims=True) + 1e-12)

    logits_t = -r_t_hybrid / temperature
    logits_t = logits_t - logits_t.max(axis=1, keepdims=True)
    gamma_t_sup = np.exp(logits_t) / (np.exp(logits_t).sum(axis=1, keepdims=True) + 1e-12)

    cluster_pred_source = gamma_s_sup.argmax(axis=1)
    cluster_pred_target = gamma_t_sup.argmax(axis=1)

    entropy_source = -np.sum(gamma_s_sup * np.log(gamma_s_sup + 1e-12), axis=1)
    entropy_target = -np.sum(gamma_t_sup * np.log(gamma_t_sup + 1e-12), axis=1)

    # 6. TẦNG 1: Lọc OOD bằng Khoảng cách Lai + Tỷ số Tách biệt Cụm gần nhì + Kiểm soát Phình cụm
    r_s_assigned = r_s_hybrid[np.arange(len(Z_s)), y_s]
    r_t_assigned = r_t_hybrid[np.arange(len(Z_t)), cluster_pred_target]

    r_t_sorted = np.sort(r_t_hybrid, axis=1)
    r_t_min1 = r_t_sorted[:, 0]
    r_t_min2 = r_t_sorted[:, 1]

    pass_separation = r_t_min1 <= (r_t_min2 * sep_ratio_thr)

    # Ước lượng Auto-Margin trên thang sqrt(r_t_assigned) từ nhóm điểm tách biệt rõ
    reliable_pool = pass_separation & (entropy_target <= np.median(entropy_target))
    if reliable_pool.sum() < 100:
        reliable_pool = pass_separation

    sqrt_r_tgt = np.sqrt(r_t_assigned[reliable_pool])
    med_t = np.median(sqrt_r_tgt)
    iqr_t = np.percentile(sqrt_r_tgt, 75.0) - np.percentile(sqrt_r_tgt, 25.0)
    global_margin = float(np.clip((med_t + 1.15 * (iqr_t / 1.349)) ** 2, 1.4, 3.8))

    # Kiểm soát phình cụm (Hubness Penalty): Nếu một cụm nhỏ trên RNA bị đám mây OOD ở trung tâm bám vào
    src_ratio = np.bincount(y_s, minlength=K) / float(len(Z_s))
    tgt_ratio = np.bincount(cluster_pred_target, minlength=K) / float(len(Z_t))

    cluster_margins = np.full(K, global_margin)
    for k in range(K):
        if tgt_ratio[k] > 1.6 * src_ratio[k]:
            penalty = ((src_ratio[k] + 0.015) / (tgt_ratio[k] + 0.015)) ** 0.45
            cluster_margins[k] = np.clip(global_margin * penalty, 1.0, global_margin)

    st_margins = cluster_margins[cluster_pred_target]
    accept_stage1 = (r_t_assigned <= st_margins) & pass_separation

    # 7. TẦNG 2: Lọc Entropy Khớp nối (Coupled Distance-Entropy)
    class_ent_thr = np.zeros(K)
    fb_ent = np.percentile(entropy_source, source_ent_pct)
    for k in range(K):
        mask_k = (y_s == k) & (r_s_assigned <= 1.0)
        if mask_k.sum() >= 5:
            class_ent_thr[k] = np.percentile(entropy_source[mask_k], source_ent_pct)
        else:
            class_ent_thr[k] = fb_ent

    src_inlier_ent = entropy_source[r_s_assigned <= 1.0]
    tgt_inlier_ent = entropy_target[accept_stage1] if accept_stage1.sum() > 50 else entropy_target
    ent_scale = float(np.clip(np.sqrt(np.maximum(np.median(tgt_inlier_ent) / (np.median(src_inlier_ent) + 1e-6), 1.0)), 1.0, 1.30))

    base_ent_thr = np.clip(class_ent_thr[cluster_pred_target] * ent_scale, 0.20, np.log(2.0))
    rel_dist = np.clip(r_t_assigned / (st_margins + 1e-12), 0.0, 1.0)
    dynamic_ent_thr = np.clip(
        base_ent_thr * (1.0 + coupling_strength * (1.0 - 2.0 * rel_dist)),
        0.15, np.log(2.0)
    )

    accept_stage2 = entropy_target <= dynamic_ent_thr
    accept_final = accept_stage1 & accept_stage2

    d2_assigned_out = d2_t_maha[np.arange(len(Z_t)), cluster_pred_target]

    info = {
        "effective_margin": round(float(st_margins.mean()), 3),
        "margin_min_max": (round(float(cluster_margins.min()), 2), round(float(cluster_margins.max()), 2)),
        "mean_ent_thr": round(float(dynamic_ent_thr.mean()), 4),
        "reject_stage1_maha": round(float((~accept_stage1).mean()), 4),
        "reject_stage2_ent": round(float((accept_stage1 & ~accept_stage2).mean()), 4),
        "raw_sup_kacc": round(float((cluster_pred_target[y_spa >= 0] == y_spa[y_spa >= 0]).mean()), 4)
    }

    return cluster_pred_target, np.sqrt(d2_assigned_out), entropy_target, accept_final, gamma_t_sup, info

# --- 1. CHẠY PHƯƠNG PHÁP SUPERVISED-PROTOTYPE SPHERICAL CASCADE ---
cl_spa, d_mah, ent_spa, acc_spa, gamma_spa_sup, info_main = supervised_prototype_spherical_cascade(
    Z_spa, Z_rna, y_rna,
    source_geom_pct=95.0, source_ent_pct=90.0,
    cosine_blend=0.65, var_shrinkage=0.20, temperature=0.15, sep_ratio_thr=0.90
)

# Cập nhật gamma_spa và label_to_cluster về ánh xạ chuẩn 1-1 để Cell vẽ 8 hình chẩn đoán hiển thị đúng Raw Argmax mới
gamma_spa = gamma_spa_sup
label_to_cluster = np.arange(K, dtype=int)

pred_names = to_names(cl_spa, acc_spa)
res_refined = open_set_metrics(to_idx(cl_spa, acc_spa), y_spa)

print(f"---> SUPERVISED-PROTOTYPE SPHERICAL CASCADE (Mean Margin = {info_main['effective_margin']} {info_main['margin_min_max']}, Mean_Ent_Thr = {info_main['mean_ent_thr']})\n")
print(f"Trần Max KACC (Raw Argmax trước khi lọc) đã tăng từ 55.4% lên: {info_main['raw_sup_kacc']:.1%}")
print(f"ST: Chấp nhận chung cuộc {acc_spa.mean():.1%} | Tổng bị loại (Unknown) {(1.0 - acc_spa.mean()):.1%}")
print(f"  ├── Bị loại ở Tầng 1 (Supervised Spherical-Maha + Sep): {info_main['reject_stage1_maha']:.1%}")
print(f"  └── Bị loại ở Tầng 2 (Coupled Distance-Entropy):        {info_main['reject_stage2_ent']:.1%}\n")

print("Số lượng tế bào phân bổ theo nhãn (ST) - Kiểm tra đủ 11 nhãn (bao gồm Endothelium & Spinal cord):")
print(pd.Series(pred_names).value_counts().to_string())

print("\n--- Đánh giá kết quả (Supervised-Prototype Spherical Cascade) ---")
print({k: round(float(v), 4) for k, v in res_refined.items()})

# --- 2. BẢNG QUÉT ĐỐI CHIẾU TÁC ĐỘNG CỦA COSINE BLEND & TỶ SỐ TÁCH BIỆT ---
bench = {}
for c_blend in (0.35, 0.50, 0.65, 0.80):
    for sep in (0.95, 0.90, 0.85):
        c, _, _, a, _, inf = supervised_prototype_spherical_cascade(
            Z_spa, Z_rna, y_rna,
            cosine_blend=c_blend, sep_ratio_thr=sep
        )
        m_res = open_set_metrics(to_idx(c, a), y_spa)
        m_res["Max_Raw_KACC"] = inf["raw_sup_kacc"]
        m_res["Margin"] = inf["effective_margin"]
        bench[f"Sup-Prototype (cos={c_blend}, sep={sep})"] = m_res

print("\n--- Bảng đối chiếu: Hiệu năng khi dùng Supervised Prototypes từ (Z_rna, y_rna) ---")
display(pd.DataFrame(bench).T.round(4))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 1. Kiểm tra và khôi phục tọa độ UMAP nếu chưa có trong bộ nhớ
if 'emb3_rna' not in globals() or 'labels_rna_plot' not in globals():
    if 'ad3' in globals() and 'is_rna' in globals():
        emb3_rna = ad3.obsm["X_umap"][is_rna]
        labels_rna_plot = np.array([class_names[i] for i in y_rna[idx_r]])
    elif 'embeddings' in globals() and 'is_rna' in globals():
        emb3_rna = embeddings["UMAP"][is_rna]
        labels_rna_plot = np.array([class_names[i] for i in y_rna[idx_r]])
    else:
        raise ValueError("Không tìm thấy biến 'ad3' hoặc 'embeddings' chứa tọa độ UMAP. Vui lòng chạy lại cell tính UMAP.")

if 'emb3_spa' not in globals():
    if 'ad3' in globals() and 'is_rna' in globals():
        emb3_spa = ad3.obsm["X_umap"][~is_rna]
    else:
        emb3_spa = embeddings["UMAP"][~is_rna]

# 2. Đồng bộ hóa chỉ số subset vẽ hình (idx_s) để tránh lệch kích thước mảng
if 'idx_s' in globals() and len(emb3_spa) == len(idx_s):
    sub_idx = idx_s
else:
    sub_idx = np.arange(len(emb3_spa))

# Trích xuất các biến dự đoán trên toàn tập ST và trên tập con vẽ hình (sub_idx)
raw_cl_spa = gamma_spa.argmax(axis=1)
raw_names_full = cluster_names[raw_cl_spa]
raw_idx_full = cluster_to_lbl[raw_cl_spa]

pred_names = to_names(cl_spa, acc_spa)
pred_idx_full = to_idx(cl_spa, acc_spa)

raw_plot = raw_names_full[sub_idx]
pred_plot = pred_names[sub_idx]
y_true_sub = y_spa[sub_idx]
acc_sub = acc_spa[sub_idx]
d_mah_sub = d_mah[sub_idx]
ent_sub = ent_spa[sub_idx]
raw_idx_sub = raw_idx_full[sub_idx]
pred_idx_sub = pred_idx_full[sub_idx]

# 3. Tạo nhãn cho Panel (2,1): Giải phẫu 5 nhóm lỗi (Error Taxonomy)
error_taxonomy = np.empty(len(sub_idx), dtype=object)
is_known_sub = y_true_sub >= 0
is_ood_sub = y_true_sub < 0

error_taxonomy[is_known_sub & acc_sub & (pred_idx_sub == y_true_sub)] = "1. Known: Đúng nhãn (TP)"
error_taxonomy[is_known_sub & acc_sub & (pred_idx_sub != y_true_sub)] = "2. Known: Gán SAI nhãn (Wrong Class)"
error_taxonomy[is_known_sub & (~acc_sub)] = "3. Known: Bị lọc nhầm (False Reject)"
error_taxonomy[is_ood_sub & acc_sub] = "4. OOD: Lọt lưới (False Accept)"
error_taxonomy[is_ood_sub & (~acc_sub)] = "5. OOD: Đúng Unknown (TN)"

error_palette = {
    "1. Known: Đúng nhãn (TP)": "#2ca02c",             # Xanh lá đậm
    "2. Known: Gán SAI nhãn (Wrong Class)": "#d62728", # Đỏ tươi
    "3. Known: Bị lọc nhầm (False Reject)": "#ff7f0e", # Cam vàng
    "4. OOD: Lọt lưới (False Accept)": "#9467bd",      # Tím đậm
    "5. OOD: Đúng Unknown (TN)": "#d3d3d3"             # Xám nhạt
}

# 4. Tạo nhãn cho Panel (2,2): Phân tách tác động của Tầng 1 và Tầng 2
e_thr_approx = info_main.get('mean_ent_thr', 0.41) if 'info_main' in globals() else 0.41
stage_status = np.empty(len(sub_idx), dtype=object)
stage_status[acc_sub] = "Chấp nhận (Pass Cả 2 Tầng)"
stage_status[(~acc_sub) & (ent_sub > e_thr_approx)] = "Bị loại do Entropy cao (Tầng 2)"
stage_status[(~acc_sub) & (ent_sub <= e_thr_approx)] = "Bị loại do Hình học/Maha (Tầng 1)"

stage_palette = {
    "Chấp nhận (Pass Cả 2 Tầng)": "#1f77b4",          # Xanh dương
    "Bị loại do Hình học/Maha (Tầng 1)": "#e377c2",   # Hồng tím
    "Bị loại do Entropy cao (Tầng 2)": "#ffbf00"      # Vàng đậm
}

# 5. Tiêu đề động cho Panel dự đoán Cascade
if 'info_main' in globals():
    m_val = info_main.get('effective_margin', 0.0)
    e_val = info_main.get('mean_ent_thr', 0.0)
    h_val = open_set_metrics(pred_idx_full, y_spa)['H-score']
    panel3_title = f"ST: Cascade (Margin={m_val:.2f}, Ent={e_val:.3f} | H={h_val:.3f})"
else:
    h_val = open_set_metrics(pred_idx_full, y_spa)['H-score']
    panel3_title = f"ST: Cascade (Accepted={acc_spa.mean():.1%} | H={h_val:.3f})"

# ==============================================================================
# KHỞI TẠO KHUNG HÌNH 2 HÀNG x 4 CỘT (8 PANELS CHẨN ĐOÁN TOÀN DIỆN)
# ==============================================================================
fig, axes = plt.subplots(2, 4, figsize=(28, 11.5))

# --- HÀNG 1: SO SÁNH NHÃN LOẠI TẾ BÀO ---
# Panel (0,0): scRNA-seq Ground Truth
scatter_cat(axes[0, 0], emb3_rna, labels_rna_plot, type_palette)
style_axis(axes[0, 0], "1. scRNA-seq: Ground Truth", "UMAP")

# Panel (0,1): ST Raw Argmax (TRƯỚC KHI LỌC UNKNOWN)
scatter_cat(axes[0, 1], emb3_spa, raw_plot, palette_unknown)
raw_kacc = (raw_idx_full[y_spa >= 0] == y_spa[y_spa >= 0]).mean()
style_axis(axes[0, 1], f"2. ST: Raw Argmax Chưa Lọc (Max KACC={raw_kacc:.1%})", "UMAP")

# Panel (0,2): ST Dự đoán sau lọc Cascade
scatter_cat(axes[0, 2], emb3_spa, pred_plot, palette_unknown)
style_axis(axes[0, 2], f"3. {panel3_title}", "UMAP")

# Panel (0,3): ST Ground Truth (OOD masked)
scatter_cat(axes[0, 3], emb3_spa, labels_spa_grouped, palette_unknown)
style_axis(axes[0, 3], "4. ST: Ground Truth (OOD masked)", "UMAP")
axes[0, 3].legend(
    loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
    markerscale=3, fontsize=8.5, title="Loại tế bào (Gồm Unknown)"
)

# --- HÀNG 2: CHẨN ĐOÁN LỖI & KHÔNG GIAN QUYẾT ĐỊNH ---
# Panel (1,0): Bản đồ Giải phẫu 5 Nhóm Lỗi
scatter_cat(axes[1, 0], emb3_spa, error_taxonomy, error_palette)
style_axis(axes[1, 0], "5. ST: Giải phẫu 5 Nhóm Lỗi (Error Map)", "UMAP")
axes[1, 0].legend(loc="lower left", frameon=True, facecolor="white", framealpha=0.85, markerscale=2.5, fontsize=8)

# Panel (1,1): Bản đồ Phân tách Tầng lọc (Stage 1 vs Stage 2)
scatter_cat(axes[1, 1], emb3_spa, stage_status, stage_palette)
style_axis(axes[1, 1], "6. ST: Tác động của Tầng 1 (Maha) vs Tầng 2 (Entropy)", "UMAP")
axes[1, 1].legend(loc="lower left", frameon=True, facecolor="white", framealpha=0.85, markerscale=2.5, fontsize=8)

# Panel (1,2): Bản đồ nhiệt Khoảng cách Mahalanobis trên UMAP
v_max = np.percentile(d_mah_sub, 95.0)
sc_hm = axes[1, 2].scatter(
    emb3_spa[:, 0], emb3_spa[:, 1], c=np.clip(d_mah_sub, 0, v_max),
    cmap="viridis", s=2.0, alpha=0.8, rasterized=True
)
plt.colorbar(sc_hm, ax=axes[1, 2], fraction=0.046, pad=0.04, label="Mahalanobis Dist (clipped p95)")
style_axis(axes[1, 2], "7. ST: Bản đồ nhiệt Khoảng cách Mahalanobis", "UMAP")

# Panel (1,3): Không gian Quyết định 2D (Mahalanobis vs. Entropy)
ax_dec = axes[1, 3]
mask_ood_p = is_ood_sub
mask_k_wrong = is_known_sub & (raw_idx_sub != y_true_sub)
mask_k_right = is_known_sub & (raw_idx_sub == y_true_sub)

ax_dec.scatter(d_mah_sub[mask_ood_p], ent_sub[mask_ood_p], c="#8c8c8c", s=2.0, alpha=0.35, label="True OOD (Unknown)", rasterized=True)
ax_dec.scatter(d_mah_sub[mask_k_wrong], ent_sub[mask_k_wrong], c="#d62728", s=2.5, alpha=0.45, label="Known (Sai nhãn ngay từ Argmax)", rasterized=True)
ax_dec.scatter(d_mah_sub[mask_k_right], ent_sub[mask_k_right], c="#2ca02c", s=2.5, alpha=0.55, label="Known (Đúng nhãn Argmax)", rasterized=True)

ax_dec.axhline(e_thr_approx, color="blue", linestyle="--", linewidth=1.2, label=f"Ngưỡng Ent TB (~{e_thr_approx:.2f})")
ax_dec.set_xlim(0, np.percentile(d_mah_sub, 98.0))
ax_dec.set_ylim(-0.02, np.log(K) * 0.75)
ax_dec.set_xlabel("Khoảng cách Mahalanobis (d_mah)")
ax_dec.set_ylabel("Shannon Entropy")
ax_dec.set_title("8. Không gian Quyết định (Maha vs. Entropy)", fontsize=10, fontweight="bold")
ax_dec.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9, markerscale=3, fontsize=8)

plt.tight_layout()
plt.savefig("fig13_diagnostic_8panels_cascade.png", bbox_inches="tight", dpi=120)
plt.show()

# ==============================================================================
# BẢNG CHẨN ĐOÁN CHI TIẾT CHO TỪNG LOẠI TẾ BÀO (GIẢI PHẪU NGUYÊN NHÂN MẤT KACC)
# ==============================================================================
rows_diag = []
for lbl_idx, c_name in enumerate(class_names):
    mask_true = (y_spa == lbl_idx)
    total_true = int(mask_true.sum())
    if total_true == 0:
        continue
    # Trước khi lọc (Raw Argmax)
    raw_correct = (raw_idx_full[mask_true] == lbl_idx).mean() * 100
    # Sau khi lọc (Cascade)
    kept_rate = acc_spa[mask_true].mean() * 100
    final_kacc_cls = (pred_idx_full[mask_true] == lbl_idx).mean() * 100
    # Nhãn bị nhầm sang nhiều nhất ở bước Raw Argmax
    wrong_preds = raw_names_full[mask_true & (raw_idx_full != lbl_idx)]
    top_confused = pd.Series(wrong_preds).value_counts().index[0] if len(wrong_preds) > 0 else "None"

    rows_diag.append({
        "True_Cell_Type": c_name,
        "Total_ST_Cells": total_true,
        "1_Raw_Argmax_Acc_%": round(raw_correct, 1),
        "2_Kept_After_Filter_%": round(kept_rate, 1),
        "3_Final_Class_KACC_%": round(final_kacc_cls, 1),
        "Most_Confused_With (Raw)": top_confused
    })

print("\n=== BẢNG GIẢI PHẪU LỖI TỪNG NHÃN: LỖI DO ARGMAX BAN ĐẦU HAY DO MÀNG LỌC CẮT NHẦM? ===")
display(pd.DataFrame(rows_diag).sort_values("Total_ST_Cells", ascending=False))

In [ ]:
km_pred = KMeans(n_clusters=n_clusters, n_init=10, random_state=CFG["seed"]).fit_predict(Z_rna)
gmm_pred = gamma_rna.argmax(1)

rows = []
for name, pred in [("K-Means trên Z", km_pred), ("GMM (argmax γ)", gmm_pred)]:
    rows.append({"Phương pháp": name,
                 "ARI": adjusted_rand_score(y_rna, pred),
                 "NMI": normalized_mutual_info_score(y_rna, pred),
                 "Số cụm dùng": len(np.unique(pred))})

metrics_df = pd.DataFrame(rows).set_index("Phương pháp").round(4)
display(metrics_df)

ax = metrics_df[["ARI", "NMI"]].plot.bar(figsize=(5.5, 3.5), color=["#3b6fb6", "#e07b39"], rot=0, width=0.7)
ax.set_ylim(0, 1); ax.set_ylabel("Score"); ax.set_title("Chất lượng phân cụm không gian ẩn (scRNA-seq)")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=9, padding=2)
plt.tight_layout(); plt.show()

In [ ]:
nn_ = NearestNeighbors(n_neighbors=31).fit(Z_plot)
_, nbr_feat = nn_.kneighbors(Z_plot)
nbr_feat = nbr_feat[:, 1:]
other_frac = (is_rna[nbr_feat] != is_rna[:, None]).mean(1)

print(f"Tỉ lệ láng giềng khác miền (trung bình): {other_frac.mean():.3f}  (lý tưởng ≈ 0.5)")
print(f"  - scRNA-seq: {other_frac[is_rna].mean():.3f} | ST: {other_frac[~is_rna].mean():.3f}")

In [ ]:
torch.save({"model": model.state_dict(), "label_mapping": label_mapping,
            "config": CFG, "hidden_dims": hidden_dims}, "gmm_vae_stage12.pt")
np.savez("latent_stage12.npz", Z_rna=Z_rna, y_rna=y_rna, gamma_rna=gamma_rna,
         Z_spa=Z_spa, gamma_spa=gamma_spa)